# Posterior analysis plan: RL vs RL-DDM
This notebook sets up a posterior comparison to assess whether RL-DDM (choices + RTs) explains the data better than RL (choices only). We’ll use:
- Posterior sampling (HMC/NUTS) for both models
- Posterior predictive checks (PPCs) for choices (RL) and choices+RTs (RL-DDM)
- Pointwise log-likelihood and PSIS-LOO (ELPD) to compare predictive performance
- Derived cross-model quantities (e.g., beta_equiv_ddm ≈ 2·a·vmod, w_hybrid) and their posteriors
Nothing runs yet; first we scaffold functions and hooks, then you can execute once generated quantities are added in Stan and sampling is enabled.

In [1]:
# Setup: imports and paths
import os
import json
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Optional: cmdstanpy (sampling later)
try:
    import cmdstanpy
except Exception as e:
    cmdstanpy = None
    print("cmdstanpy not available yet; sampling cells will be placeholders.")

# Robust project root detection: handle being launched from within 'data'
def _find_project_root(start: Path) -> Path:
    cur = start
    # If starting inside the data folder, return its parent as root
    if cur.name.lower() == 'data':
        return cur.parent
    for _ in range(8):
        if (cur / 'data').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    # fallback: if 'data' exists under start, use start; else parent of start
    return start if (start / 'data').exists() else start.parent

WORKDIR = Path.cwd()
ROOT = _find_project_root(WORKDIR)
DATA_DIR = ROOT / "data"
if not DATA_DIR.exists():
    # Absolute fallback in case root detection failed
    candidate = Path(r"C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data")
    if candidate.exists():
        DATA_DIR = candidate
        ROOT = candidate.parent
    else:
        raise FileNotFoundError(f"Could not locate data directory from {WORKDIR}. Tried {DATA_DIR} and absolute fallback.")

OUTPUTS_DIR = DATA_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"ROOT={ROOT}")
print(f"DATA_DIR={DATA_DIR}")

# CmdStan setup helper for Windows
def ensure_cmdstan_ready():
    """Ensure CmdStan is installed and toolchain is configured. Returns cmdstan_path or None."""
    if cmdstanpy is None:
        print("cmdstanpy not imported; install cmdstanpy to proceed.")
        return None
    try:
        path = cmdstanpy.cmdstan_path()
        if path and Path(path).exists():
            print(f"CmdStan found at: {path}")
            return path
    except Exception:
        pass
    # Try install CmdStan via cmdstanpy (will download and build; needs C++ toolchain)
    try:
        print("Installing CmdStan via cmdstanpy.install_cmdstan() (this may take several minutes)...")
        cmdstanpy.install_cmdstan()
        path = cmdstanpy.cmdstan_path()
        print(f"CmdStan installed at: {path}")
        return path
    except Exception as e:
        print("CmdStan install failed. Common fix on Windows: install build tools.")
        print("- Option A: Visual Studio Build Tools (C++), enable MSVC, Windows SDK")
        print("- Option B: MSYS2 with make and g++ (mingw-w64), ensure 'mingw32-make' is on PATH")
        print(f"Details: {e}")
        return None

# Utility: force-compile .stan to .exe and copy next to the .stan
def build_exe_for_model(stan_path: Path) -> Path:
    """Compile a Stan model and place the .exe next to the .stan file. Returns exe path."""
    if cmdstanpy is None:
        raise RuntimeError("cmdstanpy not available. Install and configure CmdStan first.")
    ensure_cmdstan_ready()
    print(f"Compiling: {stan_path}")
    # Force recompilation to refresh exe with current toolchain
    m = cmdstanpy.CmdStanModel(stan_file=str(stan_path), force_compile=True)
    # m.exe_file points to the compiled binary in the CmdStan install dir
    exe_src = Path(m.exe_file)
    exe_dst = stan_path.with_suffix('.exe')
    try:
        import shutil
        shutil.copy2(exe_src, exe_dst)
        print(f"Copied exe to: {exe_dst}")
        return exe_dst
    except Exception as e:
        print(f"Failed to copy exe to {exe_dst}: {e}")
        return exe_src

ROOT=c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project
DATA_DIR=c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data


In [2]:
# Helper: model sampling stubs (to be wired)
def run_sampling_stub(model_name: str, stan_file: Path, data_dict: dict, draws: int = 1000, chains: int = 4):
    """Placeholder for running cmdstanpy sampling once generated quantities exist.
    Returns a dict with mock keys expected by analysis cells."""
    return {
        "model": model_name,
        "posterior_draws": None,
        "log_lik": None,   # trial-wise log-likelihood array (draws x trials)
        "y_rep": None,      # replicated choices (RL)
        "rt_rep": None,     # replicated RTs (RL-DDM)
        "params": None,     # posterior samples dict
        "derived": None     # beta_equiv_ddm, w_hybrid, etc.
    }

def to_inferencedata_stub(result_dict):
    """Convert result to ArviZ InferenceData (placeholder)."""
    return None

In [3]:
# LOO/WAIC computation (requires pointwise log_lik)
def compute_loo_from_loglik(log_lik):
    """Compute PSIS-LOO using ArviZ from a log_lik array (draws x trials)."""
    if log_lik is None:
        print("log_lik not available; add pointwise log-likelihood in generated quantities.")
        return None
    idata = az.from_dict(log_likelihood={"y": log_lik})
    return az.loo(idata)

def summarize_loo(loo_rl, loo_rlddm):
    if loo_rl is None or loo_rlddm is None:
        print("LOO objects missing.")
        return
    print("PSIS-LOO comparison (higher ELPD is better):")
    print(f"RL     : elpd={loo_rl.elpd:.2f} ± {loo_rl.se:.2f}")
    print(f"RL-DDM : elpd={loo_rlddm.elpd:.2f} ± {loo_rlddm.se:.2f}")
    delta = loo_rlddm.elpd - loo_rl.elpd
    print(f"ΔELPD (RL-DDM − RL) = {delta:.2f}")

In [4]:
# Posterior predictive checks (PPCs)
def ppc_choices(observed_actions, y_rep_samples):
    if y_rep_samples is None:
        print("y_rep not available; simulate choices in generated quantities.")
        return
    # Example: compare mean accuracy
    obs_acc = np.mean(observed_actions == 1)
    rep_acc = np.mean(y_rep_samples == 1, axis=1)  # per draw
    plt.figure(figsize=(5,3))
    plt.hist(rep_acc, bins=30, alpha=0.6)
    plt.axvline(obs_acc, color='r', linestyle='--', label='Observed')
    plt.title('PPC: choice accuracy')
    plt.legend(); plt.show()

def ppc_rts(observed_rts, rt_rep_samples, correct_mask=None):
    if rt_rep_samples is None:
        print("rt_rep not available; simulate RTs in generated quantities.")
        return
    # Example: overlay quantiles
    qs = [0.1, 0.5, 0.9]
    obs_q = np.quantile(observed_rts, qs)
    rep_q = np.quantile(rt_rep_samples, qs, axis=1)  # draws x quantiles
    plt.figure(figsize=(6,3))
    for i,q in enumerate(qs):
        plt.hist(rep_q[:,i], bins=30, alpha=0.5, label=f'rep q{int(q*100)}')
        plt.axvline(obs_q[i], color='k', linestyle='--')
    plt.title('PPC: RT quantiles'); plt.legend(); plt.show()

In [5]:
# Derived cross-model quantities and correlations
def compute_derived(params):
    if params is None:
        print("params missing; will compute on posterior draws once available.")
        return None
    # Example structure: expects dict with arrays per draw
    vmod = params.get("vmod")
    a = params.get("a")
    beta_mb = params.get("beta_mb")
    beta_mf = params.get("beta_mf")
    if vmod is None or a is None:
        print("Need vmod and a to compute beta_equiv_ddm.")
        return None
    beta_equiv_ddm = 2 * a * vmod
    w_hybrid = None
    if beta_mb is not None and beta_mf is not None:
        w_hybrid = beta_mb / (beta_mb + beta_mf)
    return {"beta_equiv_ddm": beta_equiv_ddm, "w_hybrid": w_hybrid}

def correlate_beta2_vs_beta_equiv(beta2_draws, beta_equiv_ddm_draws):
    if beta2_draws is None or beta_equiv_ddm_draws is None:
        print("Missing draws for correlation.")
        return None
    return np.corrcoef(beta2_draws, beta_equiv_ddm_draws)[0,1]

In [6]:
# Configuration: Stan model paths and minimal data hooks
from pathlib import Path
STAN_RL = (DATA_DIR / "stan_hybrid_rlm.stan").resolve()   # RL-only model (choices)
STAN_RLDDM = (DATA_DIR / "hybrid_rl_ddm.stan").resolve()  # RL-DDM model (choices+RTs)

# Data builders wired to online_data_for_matlab.csv schema
# Expected columns: subject_id OR subject, practice_trial, trial, transition, reward, choice_1, choice_2, rt_2, state

def _prep_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Normalize column names: accept 'subject' or 'subject_id'
    subj_col = None
    for cand in ['subject', 'subject_id']:
        if cand in df.columns:
            subj_col = cand
            break
    if subj_col is None:
        raise ValueError("Missing required column: subject or subject_id")
    df = df.copy()
    df['subject'] = df[subj_col]

    # Drop practice trials if column exists, but only when 0/1 present
    if 'practice_trial' in df.columns:
        # Coerce to numeric safely
        try:
            pt = pd.to_numeric(df['practice_trial'], errors='coerce').fillna(0).astype(int)
        except Exception:
            pt = df['practice_trial']
        # If any zeros present, filter to zeros (non-practice); otherwise skip filter
        if isinstance(pt.iloc[0], (int, np.integer)) and (pt == 0).any():
            df = df[pt == 0].copy()
        else:
            print("Note: practice_trial has no 0 values (or is non-numeric); skipping practice filter.")

    # Validate essential columns and rename
    required_map = {
        'choice_1': 'c1',
        'choice_2': 'c2',
        'reward': 'r',
        'state': 'state',
        'rt_2': 'rt_2'
    }
    missing = [k for k in required_map.keys() if k not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Sort by subject and trial if available
    sort_cols = ['subject'] + ([ 'trial' ] if 'trial' in df.columns else [])
    df = df.sort_values(sort_cols)

    # Remap state: if state in {1,2}, map 1->2, 2->3; else keep
    st = df['state'].astype(int)
    if set(st.unique()) <= {1,2}:
        df['s2raw'] = st.replace({1:2, 2:3})
    else:
        df['s2raw'] = st
    # Guard: s2raw must be >=1 for Stan indexing
    s2 = pd.to_numeric(df['s2raw'], errors='coerce').fillna(1).astype(int)
    n_bad_s2 = int((s2 < 1).sum())
    if n_bad_s2:
        print(f"Warning: corrected {n_bad_s2} s2raw values <1 to 1.")
    df['s2raw'] = s2.clip(lower=1)

    # Choices: ensure 1-indexed for Stan
    # Accept 0/1 coding and map {0,1}->{1,2}; if already {1,2}, keep
    c1_raw = pd.to_numeric(df['choice_1'], errors='coerce').fillna(1).astype(int)
    c2_raw = pd.to_numeric(df['choice_2'], errors='coerce').fillna(1).astype(int)
    if set(c1_raw.unique()) <= {0,1}:
        c1_raw = c1_raw.replace({0:1, 1:2})
    if set(c2_raw.unique()) <= {0,1}:
        c2_raw = c2_raw.replace({0:1, 1:2})
    # Guard: any values <1 -> set to 1 and warn count
    n_bad_c1 = int((c1_raw < 1).sum())
    n_bad_c2 = int((c2_raw < 1).sum())
    if n_bad_c1 or n_bad_c2:
        print(f"Warning: corrected {n_bad_c1} c1 and {n_bad_c2} c2 values <1 to 1.")
        c1_raw = c1_raw.clip(lower=1)
        c2_raw = c2_raw.clip(lower=1)
    df['c1'] = c1_raw
    df['c2'] = c2_raw

    # Reward
    df['r'] = pd.to_numeric(df['reward'], errors='coerce').fillna(0).astype(float)

    # RT2 in seconds (assume ms if values >10) and enforce positive floor for DDM
    rt = pd.to_numeric(df['rt_2'], errors='coerce').astype(float)
    rt_sec = rt.where(rt < 10, rt/1000.0)
    rt_floor = 0.051  # slightly above 0.05 to ensure strict RT > Ter when Ter may be ~0.05
    n_rt_nan = int(np.isnan(rt_sec).sum())
    n_rt_nonpos = int((rt_sec <= 0).sum())
    if n_rt_nan or n_rt_nonpos:
        print(f"Warning: corrected {n_rt_nan} NaN and {n_rt_nonpos} non-positive rt_2 values to {rt_floor}s.")
    df['rt2'] = np.nan_to_num(rt_sec, nan=rt_floor)
    df['rt2'] = df['rt2'].clip(lower=rt_floor)

    # Final sanity checks
    if df.empty:
        # Provide debugging info to help diagnose
        cols = list(df.columns)
        raise ValueError("After filtering, dataframe is empty. Check practice_trial values and input CSV. Columns present: " + ", ".join(cols))
    return df


def build_data_rl(df_raw: pd.DataFrame):
    """Return a data dict for the RL-only Stan model (choices only)."""
    df = _prep_dataframe(df_raw)
    subjects = sorted(df['subject'].unique())
    S = len(subjects)
    if S == 0:
        raise ValueError("No subjects found after preprocessing.")
    groups = [df[df['subject']==s].copy() for s in subjects]
    T = [len(g) for g in groups]
    if not T or max(T) == 0:
        raise ValueError("No trials found after preprocessing.")
    T_max = max(T)
    import numpy as np
    c1 = np.ones((S, T_max), dtype=int)
    c2 = np.ones((S, T_max), dtype=int)
    r = np.zeros((S, T_max), dtype=float)
    s2raw = np.ones((S, T_max), dtype=int)
    prior_choice = np.ones(S, dtype=int)
    for i,s in enumerate(subjects):
        g = groups[i]
        prior_choice[i] = int(g.iloc[0]['c1'])
        for t, row in enumerate(g.itertuples(index=False)):
            c1[i, t] = int(row.c1)
            c2[i, t] = int(row.c2)
            r[i, t] = float(row.r)
            s2raw[i, t] = int(row.s2raw)
    data = {
        'S': S,
        'T_max': T_max,
        'T': T,
        'c1': c1,
        'c2': c2,
        'r': r,
        's2raw': s2raw,
        'prior_choice': prior_choice,
        't_common': 0.7,
    }
    print(f"Prepared RL data: S={S}, T_max={T_max}, total_trials={sum(T)}")
    return data


def build_data_rlddm(df_raw: pd.DataFrame):
    """Return a data dict for the RL-DDM Stan model (choices + RTs)."""
    df = _prep_dataframe(df_raw)
    subjects = sorted(df['subject'].unique())
    S = len(subjects)
    if S == 0:
        raise ValueError("No subjects found after preprocessing.")
    groups = [df[df['subject']==s].copy() for s in subjects]
    T = [len(g) for g in groups]
    if not T or max(T) == 0:
        raise ValueError("No trials found after preprocessing.")
    T_max = max(T)
    import numpy as np
    c1 = np.ones((S, T_max), dtype=int)
    c2 = np.ones((S, T_max), dtype=int)
    r = np.zeros((S, T_max), dtype=float)
    s2raw = np.ones((S, T_max), dtype=int)
    rt2 = np.zeros((S, T_max), dtype=float)
    prior_choice = np.ones(S, dtype=int)
    rt2_min = np.ones(S, dtype=float)
    for i,s in enumerate(subjects):
        g = groups[i]
        prior_choice[i] = int(g.iloc[0]['c1'])
        rt2_min[i] = float(np.min(g['rt2']))
        for t, row in enumerate(g.itertuples(index=False)):
            c1[i, t] = int(row.c1)
            c2[i, t] = int(row.c2)
            r[i, t] = float(row.r)
            s2raw[i, t] = int(row.s2raw)
            rt2[i, t] = float(row.rt2)
    data = {
        'S': S,
        'T_max': T_max,
        'T': T,
        'c1': c1,
        'c2': c2,
        'r': r,
        's2raw': s2raw,
        'rt2': rt2,
        'rt2_min': rt2_min,  # per-subject minimum RT for potential Ter upper-bounds in Stan
        'rt2_min_global': float(np.min(rt2_min)),
        'ter_eps': 0.001,     # small epsilon to keep Ter strictly below rt2_min
        'prior_choice': prior_choice,
        't_common': 0.7,
    }
    print(f"Prepared RL-DDM data: S={S}, T_max={T_max}, total_trials={sum(T)}; rt2_min_global={data['rt2_min_global']:.3f}s")
    return data

In [7]:
# Sampling runner: swap stubs with real calls once ready
def run_sampling(
    model_path: Path,
    data_dict: dict,
    draws: int = 1000,
    chains: int = 4,
    warmup: int | None = None,
    adapt_delta: float | None = None,
    max_treedepth: int | None = None,
    seed: int | None = None,
    output_dir: Path | None = None,
    inits: dict | None = None,
):
    if cmdstanpy is None:
        print("cmdstanpy not available; install and configure cmdstan before sampling.")
        return run_sampling_stub(model_path.name, model_path, data_dict, draws, chains)
    # Prefer precompiled exe next to the .stan file if present
    exe_path = model_path.with_suffix('.exe')
    try:
        if exe_path.exists():
            print(f"Using precompiled model exe: {exe_path}")
            m = cmdstanpy.CmdStanModel(stan_file=str(model_path), exe_file=str(exe_path), compile=False)
        else:
            print("No precompiled exe found; compiling with CmdStan...")
            m = cmdstanpy.CmdStanModel(stan_file=str(model_path))
        # Ensure logs go to a stable folder for debugging
        run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        if output_dir is None:
            out_base = OUTPUTS_DIR / "cmdstan_runs" / f"{model_path.stem}_{run_ts}"
        else:
            out_base = Path(output_dir)
        out_base.mkdir(parents=True, exist_ok=True)
        print(f"CmdStan output_dir: {out_base}")
        sample_kwargs = {
            "data": data_dict,
            "iter_sampling": draws,
            "chains": chains,
            "show_console": True,
            "refresh": 100,
            "output_dir": str(out_base),
        }
        if warmup is not None:
            sample_kwargs["iter_warmup"] = warmup
            print(f"Using custom warmup: iter_warmup={warmup}")
        if adapt_delta is not None:
            sample_kwargs["adapt_delta"] = adapt_delta
            print(f"Using adapt_delta={adapt_delta}")
        if max_treedepth is not None:
            sample_kwargs["max_treedepth"] = max_treedepth
            print(f"Using max_treedepth={max_treedepth}")
        if seed is not None:
            sample_kwargs["seed"] = seed
            print(f"Using seed={seed}")
        # Provide safe initial values to avoid rt <= Ter errors in DDM
        if inits is None and isinstance(data_dict, dict) and "rt2" in data_dict:
            import numpy as _np
            # Base Ter from global rt2 minimum
            rt2 = _np.array(data_dict["rt2"])  # S x T_max
            rt_min = float(_np.min(rt2[_np.isfinite(rt2)])) if rt2.size else 0.05
            ter_base = 0.05 if not _np.isfinite(rt_min) or rt_min <= 0.06 else max(0.05, rt_min * 0.5)
            # Vectorized per-subject inits using rt2_min to ensure Ter < rt2_min - ter_eps
            S = int(data_dict.get("S", _np.shape(rt2)[0]))
            rt2_min = _np.array(data_dict.get("rt2_min", [ter_base] * S), dtype=float)
            eps = float(data_dict.get("ter_eps", 1e-3))
            ter_vec = []
            for v in rt2_min:
                # pick a safe value strictly below v - eps, and above a small floor
                safe = min(max(0.03, ter_base), float(v) - max(eps, 1e-3))
                # if v is tiny, fallback to 0.03
                if not _np.isfinite(safe) or safe <= 0:
                    safe = 0.03
                ter_vec.append(safe)
            inits = {"Ter": ter_vec}
            print(f"Auto inits: Ter per-subject len={len(ter_vec)}, sample (first 5)={ter_vec[:5]}")
        if inits is not None:
            sample_kwargs["inits"] = inits
        fit = m.sample(**sample_kwargs)
    except Exception as e:
        print(f"Model build/run failed: {e}")
        print("Tip: Logs were written under the output_dir above; check *-stdout.txt for the precise error.")
        print("You can also run the 'Debug CmdStan logs' cell below to print the last lines here.")
        raise
    # Extract basics; replace None with actual arrays once GQ exists
    result = {
        "model": model_path.name,
        "posterior_draws": fit.draws(),
        "log_lik": None,  # e.g., fit.draws_pd()["log_lik"] reshaped (draws x trials)
        "y_rep": None,     # e.g., fit.draws_pd()[colnames for y_rep]
        "rt_rep": None,    # e.g., fit.draws_pd()[colnames for rt_rep]
        "params": None,    # pack parameter draws as needed
        "derived": None     # beta_equiv_ddm, w_hybrid, etc.
    }
    return result

In [8]:
# Execute: sample both models (placeholder until data builders return dicts)
def run_both_models(draws=1000, chains=4):
    data_rl = build_data_rl()
    data_rlddm = build_data_rlddm()
    rl = run_sampling(STAN_RL, data_rl, draws=draws, chains=chains)
    rlddm = run_sampling(STAN_RLDDM, data_rlddm, draws=draws, chains=chains)
    return rl, rlddm

# After sampling, compute LOO and show comparison
def analyze_models(rl_result, rlddm_result):
    loo_rl = compute_loo_from_loglik(rl_result.get("log_lik"))
    loo_rlddm = compute_loo_from_loglik(rlddm_result.get("log_lik"))
    summarize_loo(loo_rl, loo_rlddm)
    # PPCs (provide observed data when wiring)
    # ppc_choices(observed_actions, rl_result.get("y_rep"))
    # ppc_rts(observed_rts, rlddm_result.get("rt_rep"), correct_mask)
    # Derived comparisons
    # d_rl = compute_derived(rl_result.get("params"))
    # d_rlddm = compute_derived(rlddm_result.get("params"))
    # print(correlate_beta2_vs_beta_equiv(beta2_draws, d_rlddm["beta_equiv_ddm"]))

In [9]:
# Usage guide
print("Steps to enable full posterior analysis:")
print("1) In your Stan files (RL, RL-DDM), add generated quantities: trial-wise log_lik, y_rep (RL), rt_rep (RL-DDM), and derived quantities (beta_equiv_ddm, w_hybrid).")
print("2) Implement build_data_rl() and build_data_rlddm() to feed the same data you use in stan_test.ipynb.")
print("3) Ensure cmdstanpy is installed and CmdStan is built; then run run_both_models().")
print("4) Call analyze_models(rl_result, rlddm_result) to compute LOO and render PPCs.")

Steps to enable full posterior analysis:
1) In your Stan files (RL, RL-DDM), add generated quantities: trial-wise log_lik, y_rep (RL), rt_rep (RL-DDM), and derived quantities (beta_equiv_ddm, w_hybrid).
2) Implement build_data_rl() and build_data_rlddm() to feed the same data you use in stan_test.ipynb.
3) Ensure cmdstanpy is installed and CmdStan is built; then run run_both_models().
4) Call analyze_models(rl_result, rlddm_result) to compute LOO and render PPCs.


In [10]:
# Extraction helpers: map fit outputs to analysis inputs
def extract_arrays_from_fit(fit, trial_count):
    """Given a cmdstanpy fit, extract arrays needed for LOO/PPC.
    Implement column naming to match your generated quantities."""
    if fit is None:
        return {"log_lik": None, "y_rep": None, "rt_rep": None, "params": None, "derived": None}
    df = fit.draws_pd()
    # Example column names; update to actual names produced by Stan
    # log_lik columns often look like 'log_lik[1]', 'log_lik[2]', ...
    loglik_cols = [f"log_lik[{i}]" for i in range(1, trial_count+1)]
    loglik = df[loglik_cols].to_numpy()  # shape: draws x T
    # y_rep (discrete): 'y_rep[1]' ...
    yrep_cols = [f"y_rep[{i}]" for i in range(1, trial_count+1) if f"y_rep[{i}]" in df.columns]
    yrep = df[yrep_cols].to_numpy() if yrep_cols else None
    # rt_rep (continuous): 'rt_rep[1]' ...
    rtrep_cols = [f"rt_rep[{i}]" for i in range(1, trial_count+1) if f"rt_rep[{i}]" in df.columns]
    rtrep = df[rtrep_cols].to_numpy() if rtrep_cols else None
    # Parameter draws: pack a few of interest if present
    params = {}
    for pname in ["beta_mb", "beta_mf", "beta2", "vmod", "a", "z", "Ter"]:
        if pname in df.columns:
            params[pname] = df[pname].to_numpy()
    # Derived draws
    derived = {}
    for dname in ["beta_equiv_ddm", "w_hybrid"]:
        if dname in df.columns:
            derived[dname] = df[dname].to_numpy()
    return {"log_lik": loglik, "y_rep": yrep, "rt_rep": rtrep, "params": params, "derived": derived}

In [11]:
# Data loading and sampling execution
def load_data_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    # Expect columns: subject,c1,c2,r,s2raw,rt2
    required = {"subject","c1","c2","r","s2raw","rt2"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing columns: {missing}")
    return df


In [12]:
# Run posterior sampling end-to-end on final_dataset.csv (quick test run, hardened inits)
from pathlib import Path
csv_path = Path(r"C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\final_dataset.csv")
print(f"Loading: {csv_path}")
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found at {csv_path}. Check path.")
df_all = pd.read_csv(csv_path)
rl_data = build_data_rl(df_all)
rlddm_data = build_data_rlddm(df_all)

# Sanity checks before sampling
import numpy as np
min_s2_rl = int(np.min(rl_data['s2raw']))
min_s2_rlddm = int(np.min(rlddm_data['s2raw']))
rt2_min_global = float(rlddm_data.get('rt2_min_global', np.min(rlddm_data['rt2'])))
print(f"Sanity: min s2raw (RL)={min_s2_rl}, (RL-DDM)={min_s2_rlddm}; rt2_min_global={rt2_min_global:.3f}s")
if min_s2_rl < 1 or min_s2_rlddm < 1:
    raise ValueError(f"s2raw contains values <1 after preprocessing: RL min={min_s2_rl}, RL-DDM min={min_s2_rlddm}. Please re-run the data prep cell (Cell 7).")

# Ensure CmdStan is installed and ready before sampling
cmdstan_path = ensure_cmdstan_ready()
if cmdstan_path is None:
    raise RuntimeError("CmdStan not ready. Please install Windows C++ build tools or MSYS2/mingw and re-run ensure_cmdstan_ready().")

# Hardened test run settings to reduce mid-run failures
quick_draws = 200
quick_chains = 2
quick_warmup = 500  # extra warmup for more stable adaptation
print(f"Quick test (hardened): draws={quick_draws}, chains={quick_chains}, warmup={quick_warmup}")

# Sample (hardened init & controls)
import inspect
rs_sig = inspect.signature(run_sampling)
call_kwargs = dict(draws=quick_draws, chains=quick_chains, adapt_delta=0.99, max_treedepth=12, seed=9970)
if 'warmup' in rs_sig.parameters:
    call_kwargs['warmup'] = quick_warmup

# Provide explicit safe inits for DDM params: Ter (per subject), a, z, vmod
rtmins = np.array(rlddm_data['rt2_min'], dtype=float)
eps = float(rlddm_data.get('ter_eps', 0.001))
# Keep Ter strictly below each subject's rt2_min with added margin
Ter_inits = []
for v in rtmins:
    # Use margin = eps + 0.02 to be conservative; clamp to floor 0.03
    safe_ter = max(0.03, float(v) - (eps + 0.020))
    # If rt2_min is extremely small (close to floor), back off further
    if safe_ter >= (float(v) - eps):
        safe_ter = max(0.03, float(v) * 0.5)
    Ter_inits.append(safe_ter)

# Choose interior values for other constrained params
a_init = 1.8   # moderate boundary separation
z_init = 0.5   # centered starting bias
vmod_init = 1.0  # positive drift scaling

inits = {
    'Ter': Ter_inits,
    'a': a_init,
    'z': z_init,
    'vmod': vmod_init,
}
print(f"Init settings: Ter len={len(Ter_inits)}, sample (first 5)={Ter_inits[:5]}, a={a_init}, z={z_init}, vmod={vmod_init}")
call_kwargs['inits'] = inits

#rl_result = run_sampling(STAN_RL, rl_data, **call_kwargs)
#rlddm_result = run_sampling(STAN_RLDDM, rlddm_data, **call_kwargs)

# Analyze (LOO + PPC placeholders)
#analyze_models(rl_result, rlddm_result)

Loading: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\final_dataset.csv
Prepared RL data: S=151, T_max=200, total_trials=30200
Prepared RL-DDM data: S=151, T_max=200, total_trials=30200; rt2_min_global=0.051s
Sanity: min s2raw (RL)=1, (RL-DDM)=1; rt2_min_global=0.051s
CmdStan found at: C:\Users\Taychaz\.cmdstan\cmdstan-2.37.0
Quick test (hardened): draws=200, chains=2, warmup=500
Init settings: Ter len=151, sample (first 5)=[0.03, 0.35506500001857055, 0.03, 0.19243000000342728, 0.03], a=1.8, z=0.5, vmod=1.0


In [13]:
# Debug CmdStan logs: print last lines of most recent run
from pathlib import Path
from glob import glob
import os

def _tail_file(path: Path, n_lines: int = 80):
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        for line in lines[-n_lines:]:
            print(line.rstrip())
    except Exception as e:
        print(f"Failed to read {path}: {e}")

def print_last_cmdstan_logs(n_lines: int = 80):
    base = OUTPUTS_DIR / 'cmdstan_runs'
    if not base.exists():
        print(f"No cmdstan_runs folder yet at {base}")
        return
    # Find most recent subfolder
    subdirs = [Path(p) for p in glob(str(base / '*')) if os.path.isdir(p)]
    if not subdirs:
        print(f"No run folders found under {base}")
        return
    latest = max(subdirs, key=os.path.getmtime)
    print(f"Latest run folder: {latest}")
    # Print stdout files (all chains)
    stdout_files = sorted(Path(latest).glob('*-stdout.txt'))
    if not stdout_files:
        print("No console logs found (files *-stdout.txt)")
        return
    for fp in stdout_files:
        print("\n==> ", fp)
        _tail_file(fp, n_lines=n_lines)
print_last_cmdstan_logs(120)

Latest run folder: c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\hybrid_rl_ddm_20251202_202439

==>  c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\hybrid_rl_ddm_20251202_202439\hybrid_rl_ddm-20251202202439_0-stdout.txt
method = sample (Default)
  sample
    num_samples = 50
    num_warmup = 150
    save_warmup = false (Default)
    thin = 1 (Default)
    adapt
      engaged = true (Default)
      gamma = 0.05 (Default)
      delta = 0.995
      kappa = 0.75 (Default)
      t0 = 10 (Default)
      init_buffer = 75 (Default)
      term_buffer = 50 (Default)
      window = 25 (Default)
      save_metric = false (Default)
    algorithm = hmc (Default)
      hmc
        engine = nuts (Default)
          nuts
            max_depth = 12
        metric = diag_e (Default)
        metric_file =  (Default)
        stepsize = 1 (Default)
        stepsize_jitter = 0 (Default)
    num_chains = 1 (Default)
id = 1 (Default)
data
  file = 

In [15]:
# Compile Stan models to .exe (Windows-friendly) with MSYS2 PATH preflight
import os, shutil, time

# Prepend MSYS2 bins to PATH for this process (no system changes).
msys_bins = [r"C:\\msys64\\ucrt64\\bin", r"C:\\msys64\\usr\\bin"]
for p in msys_bins:
    try:
        if os.path.isdir(p) and p not in os.environ.get("PATH", ""):
            os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]
    except Exception:
        pass
print("Ensured MSYS2 bins are at PATH front:", "; ".join([p for p in msys_bins if p in os.environ.get("PATH", "")] ))

# Verify required tools are available
required_tools = ["mingw32-make", "g++", "cut", "expr"]
missing = [t for t in required_tools if shutil.which(t) is None]
if missing:
    raise RuntimeError(
        "Missing tools on PATH: " + ", ".join(missing) + ". "
        "Add C\\msys64\\ucrt64\\bin AND C\\msys64\\usr\\bin to PATH, then fully restart VS Code."
    )
for t in required_tools:
    print(f"Found {t} at: {shutil.which(t)}")

print("Compiling Stan models (this may take a minute)...")
cmdstan_path = ensure_cmdstan_ready()
if cmdstan_path is None:
    raise RuntimeError("CmdStan not ready. Install toolchain and re-run ensure_cmdstan_ready().")

# Build RL and RL-DDM executables next to their .stan files
def _compile_with_retry(stan_path, retries=1, delay=1.0):
    try:
        return build_exe_for_model(stan_path)
    except Exception as e:
        if retries > 0:
            print(f"Compile/copy failed once for {stan_path.name}: {e}. Retrying in {delay}s...")
            time.sleep(delay)
            return _compile_with_retry(stan_path, retries=retries-1, delay=delay*2)
        raise

try:
    rl_exe = _compile_with_retry(STAN_RL)
    print(f"RL exe: {rl_exe}")
except Exception as e:
    print(f"Failed to compile RL model: {e}")

try:
    rlddm_exe = _compile_with_retry(STAN_RLDDM)
    print(f"RL-DDM exe: {rlddm_exe}")
except Exception as e:
    print(f"Failed to compile RL-DDM model: {e}")

print("Compilation step finished. Proceed to the sampling cell.")

Ensured MSYS2 bins are at PATH front: C:\\msys64\\ucrt64\\bin; C:\\msys64\\usr\\bin
Found mingw32-make at: C:\\msys64\\ucrt64\\bin\mingw32-make.EXE
Found g++ at: C:\\msys64\\ucrt64\\bin\g++.EXE
Found cut at: C:\\msys64\\usr\\bin\cut.EXE
Found expr at: C:\\msys64\\usr\\bin\expr.EXE
Compiling Stan models (this may take a minute)...
CmdStan found at: C:\Users\Taychaz\.cmdstan\cmdstan-2.37.0
CmdStan found at: C:\Users\Taychaz\.cmdstan\cmdstan-2.37.0
Compiling: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\stan_hybrid_rlm.stan
Failed to copy exe to C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\stan_hybrid_rlm.exe: [WinError 32] The process cannot access the file because it is being used by another process
RL exe: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\stan_hybrid_rlm.exe
CmdStan found at: C:\Users\Taychaz\.cmdstan\cmdstan-2.37.0
Compiling: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.stan


18:32:24 - cmdstanpy - INFO - compiling stan file C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.stan to exe file C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.exe
18:32:36 - cmdstanpy - INFO - compiled model executable: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.exe
18:32:36 - cmdstanpy - INFO - compiled model executable: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.exe


Failed to copy exe to C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.exe: [WinError 32] The process cannot access the file because it is being used by another process
RL-DDM exe: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\hybrid_rl_ddm.exe
Compilation step finished. Proceed to the sampling cell.


In [22]:
# Rebuild CmdStan with MSYS2 g++ and dynamic libstdc++ (fixes linker mismatch)
import os, subprocess, shutil
from pathlib import Path

# Ensure MSYS2 bins are on PATH for this process
msys_bins = [r"C:\\msys64\\ucrt64\\bin", r"C:\\msys64\\usr\\bin"]
for p in msys_bins:
    if os.path.isdir(p) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]
print("MSYS2 PATH prepared:", "; ".join([p for p in msys_bins if p in os.environ.get("PATH", "")] ))

if cmdstanpy is None:
    raise RuntimeError("cmdstanpy not available. Install it first.")

cmdstan_path = Path(cmdstanpy.cmdstan_path()).resolve()
print(f"CmdStan path: {cmdstan_path}")

# Write/override make/local to avoid static libstdc++ (reduces ABI/linker issues)
# and ensure g++/gcc from MSYS2 are used.
make_local = cmdstan_path / "make" / "local"
make_local.parent.mkdir(parents=True, exist_ok=True)
local_lines = [
    "CXX=g++",
    "CC=gcc",
    "STATIC_LIBSTDCXX=false",
]
make_local.write_text("\n".join(local_lines) + "\n", encoding="ascii")
print(f"Wrote {make_local} with:\n" + "\n".join(local_lines))

# Verify tools
for t in ["mingw32-make", "g++", "gcc"]:
    path = shutil.which(t)
    if not path:
        raise RuntimeError(f"Required tool not found on PATH: {t}")
    print(f"Found {t} at: {path}")

# Clean and rebuild CmdStan
print("Cleaning CmdStan (this may take a minute)...")
res_clean = subprocess.run(["mingw32-make", "-C", str(cmdstan_path), "clean-all"], capture_output=True, text=True)
if res_clean.returncode != 0:
    print(res_clean.stdout)
    print(res_clean.stderr)
    raise RuntimeError("CmdStan clean-all failed.")
print("Clean complete. Building CmdStan...")
res_build = subprocess.run(["mingw32-make", "-C", str(cmdstan_path), "build"], capture_output=True, text=True)
if res_build.returncode != 0:
    print(res_build.stdout)
    print(res_build.stderr)
    raise RuntimeError("CmdStan build failed. See output above for details.")

print("CmdStan successfully rebuilt with MSYS2 g++ and dynamic libstdc++. Now re-run the compile cell above.")

MSYS2 PATH prepared: C:\\msys64\\ucrt64\\bin; C:\\msys64\\usr\\bin
CmdStan path: C:\Users\Taychaz\.cmdstan\cmdstan-2.37.0
Wrote C:\Users\Taychaz\.cmdstan\cmdstan-2.37.0\make\local with:
CXX=g++
CC=gcc
STATIC_LIBSTDCXX=false
Found mingw32-make at: C:\\msys64\\ucrt64\\bin\mingw32-make.EXE
Found g++ at: C:\\msys64\\ucrt64\\bin\g++.EXE
Found gcc at: C:\\msys64\\ucrt64\\bin\gcc.EXE
Cleaning CmdStan (this may take a minute)...
Clean complete. Building CmdStan...
CmdStan successfully rebuilt with MSYS2 g++ and dynamic libstdc++. Now re-run the compile cell above.


In [ ]:
# Diagnostics: check divergences and treedepth from CmdStan CSVs
import pandas as pd
import numpy as np
from glob import glob
import os

def summarize_cmdstan_run(run_dir=None):
    if run_dir is None:
        # Find latest run in outputs/cmdstan_runs
        base = OUTPUTS_DIR / 'cmdstan_runs'
        if not base.exists():
            print("No cmdstan_runs folder found.")
            return
        subdirs = [Path(p) for p in glob(str(base / '*')) if os.path.isdir(p)]
        if not subdirs:
            print("No run folders found.")
            return
        run_dir = max(subdirs, key=os.path.getmtime)
    
    print(f"Analyzing run: {run_dir}")
    csv_files = sorted(Path(run_dir).glob('*-*.csv'))
    if not csv_files:
        print("No CSV files found in run dir.")
        return

    total_draws = 0
    divergences = 0
    max_td_seen = 0
    
    print(f"Found {len(csv_files)} chains.")
    for i, csv in enumerate(csv_files):
        try:
            # Read only comment lines to find config, then read data
            # CmdStanPy CSVs have comments then header
            try:
                df = pd.read_csv(csv, comment='#')
            except pd.errors.EmptyDataError:
                print(f"Chain {i+1}: Empty file (no data).")
                continue
                
            if df.empty:
                print(f"Chain {i+1}: Empty dataframe.")
                continue

            # Robust column access
            cols = df.columns
            # divergence is 'divergent__'
            if 'divergent__' in cols:
                divs = df['divergent__'].sum()
                divergences += divs
            else:
                divs = 0
            
            # treedepth is 'treedepth__'
            if 'treedepth__' in cols:
                # Force numeric, coercing errors to NaN (handles object types)
                td = pd.to_numeric(df['treedepth__'], errors='coerce')
                # Handle NaNs safely
                td_valid = td[np.isfinite(td)]
                if not td_valid.empty:
                    max_td_seen = max(max_td_seen, int(td_valid.max()))
            
            total_draws += len(df)
            
            # Report per chain
            print(f"Chain {i+1}: {len(df)} draws, {divs if 'divergent__' in cols else '?'} divergences")
            
        except Exception as e:
            print(f"Error reading chain {i+1} ({csv.name}): {e}")

    print("-" * 40)
    print(f"Total draws: {total_draws}")
    print(f"Total divergences: {divergences}")
    print(f"Max treedepth seen: {max_td_seen}")
    if divergences > 0:
        print("WARNING: Divergences detected. Consider increasing adapt_delta (e.g., to 0.99 or 0.995).")
    else:
        print("No divergences detected.")


Analyzing run: c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\quick_check_20251202_211731
Found 2 chains.
Chain 1: Empty dataframe.
Chain 2: Empty dataframe.
----------------------------------------
Total draws: 0
Total divergences: 0
Max treedepth seen: 0
No divergences detected.


In [25]:
# Conservative quick init-check variant to isolate instability
import numpy as np
from pathlib import Path
import cmdstanpy

csv_path = Path(r"C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\final_dataset.csv")
print(f"Loading for quick check: {csv_path}")
df_all = pd.read_csv(csv_path)
rlddm_data = build_data_rlddm(df_all)
S = int(rlddm_data['S'])
rtmins = np.array(rlddm_data['rt2_min'], dtype=float)
eps = float(rlddm_data.get('ter_eps', 0.001))

# Stricter Ter margin
Ter_inits = []
for v in rtmins:
    safe_ter = max(0.03, float(v) - (eps + 0.050))
    if safe_ter >= (float(v) - eps):
        safe_ter = max(0.03, float(v) * 0.5)
    Ter_inits.append(safe_ter)

# More interior parameter inits
inits = {
    'Ter': Ter_inits,
    'a': [1.5] * S,    # smaller boundary
    'z': [0.5] * S,    # center
    'vmod': [0.6] * S, # smaller drift scaling
}
print(f"Inits: Ter[0:5]={Ter_inits[:5]} a[0]={inits['a'][0]} z[0]={inits['z'][0]} vmod[0]={inits['vmod'][0]} S={S}")

# Model and exe
model_path = STAN_RLDDM
exe_path = model_path.with_suffix('.exe')
m = cmdstanpy.CmdStanModel(
    stan_file=str(model_path),
    exe_file=str(exe_path) if exe_path.exists() else None,
    compile=False if exe_path.exists() else True
)

# Output dir
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = OUTPUTS_DIR / "cmdstan_runs" / f"quick_check_conservative_{run_ts}"
out_dir.mkdir(parents=True, exist_ok=True)

fit = m.sample(
    data=rlddm_data,
    iter_sampling=40,
    iter_warmup=200,
    chains=1,
    adapt_delta=0.995,
    max_treedepth=12,
    seed=9001,
    inits=inits,
    step_size=0.03,
    output_dir=str(out_dir),
    show_console=True,
    refresh=10,
)
print("Conservative quick check done.")
summarize_cmdstan_run(out_dir)

Loading for quick check: C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\final_dataset.csv


21:57:50 - cmdstanpy - WARNING - CmdStanModel(compile=...) is deprecated and will be removed in the next major version. The constructor will always ensure a model has a compiled executable.
If you wish to force recompilation, use force_compile=True instead.
21:57:50 - cmdstanpy - INFO - Chain [1] start processing


Prepared RL-DDM data: S=151, T_max=200, total_trials=30200; rt2_min_global=0.051s
Inits: Ter[0:5]=[0.03, 0.3250650000185706, 0.03, 0.16243000000342728, 0.03] a[0]=1.5 z[0]=0.5 vmod[0]=0.6 S=151
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 40
Chain [1] num_warmup = 200
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.995
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 12
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 0.03
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (D

22:10:06 - cmdstanpy - INFO - Chain [1] done processing
22:10:06 - cmdstanpy - ERROR - Chain [1] error: code '3221225477' 


Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 

RuntimeError: Error during sampling:

Command and output files:
RunSet: chains=1, chain_ids=[1], num_processes=1
 cmd (chain 1):
	['C:\\Users\\Taychaz\\Desktop\\Oguz\\Thesis\\thesis-project\\data\\hybrid_rl_ddm.exe', 'id=1', 'random', 'seed=9001', 'data', 'file=C:\\Users\\Taychaz\\AppData\\Local\\Temp\\tmp1hmaucbo\\rg34clix.json', 'init=C:\\Users\\Taychaz\\AppData\\Local\\Temp\\tmp1hmaucbo\\1a5opsp5.json', 'output', 'file=C:\\Users\\Taychaz\\Desktop\\Oguz\\Thesis\\thesis-project\\data\\outputs\\cmdstan_runs\\quick_check_conservative_20251202_215750\\hybrid_rl_ddm-20251202215750.csv', 'refresh=10', 'method=sample', 'num_samples=40', 'num_warmup=200', 'algorithm=hmc', 'engine=nuts', 'max_depth=12', 'stepsize=0.03', 'adapt', 'engaged=1', 'delta=0.995']
 retcodes=[3221225477]
 per-chain output files (showing chain 1 only):
 csv_file:
	C:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\quick_check_conservative_20251202_215750\hybrid_rl_ddm-20251202215750.csv
 cons

In [21]:
# Tail the latest run logs
from pathlib import Path
from glob import glob
import os

def tail(path, n=300):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    for line in lines[-n:]:
        print(line.rstrip())

base = OUTPUTS_DIR / 'cmdstan_runs'
latest = None
if base.exists():
    subdirs = [Path(p) for p in glob(str(base / '*')) if os.path.isdir(p)]
    if subdirs:
        latest = max(subdirs, key=os.path.getmtime)
print(f"Latest run: {latest}")
if latest:
    for fp in sorted(Path(latest).glob('*-stdout.txt')):
        print("\n=== LOG:", fp.name, "===\n")
        tail(fp, n=300)

Latest run: c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\quick_check_conservative_20251202_214149

=== LOG: hybrid_rl_ddm-20251202214149_0-stdout.txt ===

method = sample (Default)
  sample
    num_samples = 40
    num_warmup = 200
    save_warmup = false (Default)
    thin = 1 (Default)
    adapt
      engaged = true (Default)
      gamma = 0.05 (Default)
      delta = 0.995
      kappa = 0.75 (Default)
      t0 = 10 (Default)
      init_buffer = 75 (Default)
      term_buffer = 50 (Default)
      window = 25 (Default)
      save_metric = false (Default)
    algorithm = hmc (Default)
      hmc
        engine = nuts (Default)
          nuts
            max_depth = 12
        metric = diag_e (Default)
        metric_file =  (Default)
        stepsize = 0.05
        stepsize_jitter = 0 (Default)
    num_chains = 1 (Default)
id = 1 (Default)
data
  file = C:\Users\Taychaz\AppData\Local\Temp\tmp1hmaucbo\kn421dzf.json
init = C:\Users\Taychaz\AppData\Local\Te

In [18]:
summarize_cmdstan_run(out_dir)

Analyzing run: c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\quick_check_20251202_211731
Found 2 chains.
Chain 1: Empty dataframe.
Chain 2: Empty dataframe.
----------------------------------------
Total draws: 0
Total divergences: 0
Max treedepth seen: 0
No divergences detected.


In [19]:
# Inspect quick check logs for the last run
from pathlib import Path
from glob import glob
import os

def tail(path, n=200):
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        for line in lines[-n:]:
            print(line.rstrip())
    except Exception as e:
        print(f"Failed reading {path}: {e}")

base = OUTPUTS_DIR / 'cmdstan_runs'
latest = None
if base.exists():
    subdirs = [Path(p) for p in glob(str(base / '*')) if os.path.isdir(p)]
    if subdirs:
        latest = max(subdirs, key=os.path.getmtime)

print(f"Latest run: {latest}")
if latest:
    for fp in sorted(Path(latest).glob('*-stdout.txt')):
        print("\n=== LOG:", fp.name, "===\n")
        tail(fp, n=300)

Latest run: c:\Users\Taychaz\Desktop\Oguz\Thesis\thesis-project\data\outputs\cmdstan_runs\quick_check_20251202_211731

=== LOG: hybrid_rl_ddm-20251202211731_0-stdout.txt ===

method = sample (Default)
  sample
    num_samples = 50
    num_warmup = 150
    save_warmup = false (Default)
    thin = 1 (Default)
    adapt
      engaged = true (Default)
      gamma = 0.05 (Default)
      delta = 0.995
      kappa = 0.75 (Default)
      t0 = 10 (Default)
      init_buffer = 75 (Default)
      term_buffer = 50 (Default)
      window = 25 (Default)
      save_metric = false (Default)
    algorithm = hmc (Default)
      hmc
        engine = nuts (Default)
          nuts
            max_depth = 12
        metric = diag_e (Default)
        metric_file =  (Default)
        stepsize = 0.1
        stepsize_jitter = 0 (Default)
    num_chains = 1 (Default)
id = 1 (Default)
data
  file = C:\Users\Taychaz\AppData\Local\Temp\tmp1hmaucbo\du6pwprf.json
init = C:\Users\Taychaz\AppData\Local\Temp\tmp1hmaucbo